# Libraries

In [1]:
import os
import random
import numpy as np
from PIL import Image
from tqdm import tqdm
import pydicom
import pandas as pd
from pydicom.multival import MultiValue
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import math
from torchvision import transforms, models
import csv
import timm
import torch.nn as nn
import torch.nn.functional as F
# optional: for AUC
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, matthews_corrcoef, roc_curve,
    brier_score_loss, balanced_accuracy_score, accuracy_score
)

In [2]:
# TRAIN_CSV = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_train.csv"
# VAL_CSV   = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_val.csv"
# TEST_CSV  = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test.csv"

TRAIN_CSV = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_train_reduced.csv"
VAL_CSV   = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_val_reduced.csv"
TEST_CSV  = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_test_reduced.csv"


# ===============================
# Task configuration
# ===============================
# NUM_CLASSES = 4
# CLASS_NAMES = ["NORMAL", "DRUSEN", "DME", "CNV"]

NUM_CLASSES = 2
CLASS_NAMES = ["NORMAL", "DISEASE"]


# Label meaning:
# four_class_label:
# 0 -> NORMAL
# 1 -> DRUSEN
# 2 -> DME
# 3 -> CNV

# binary_label:
# 0 -> NORMAL
# 1 -> DISEASE (DRUSEN / DME / CNV)

# ===============================
# Training configuration (initial)
# ===============================
IMG_SIZE = 512        # standard for ResNet18
BATCH_SIZE = 32
NUM_WORKERS = 4
NUM_EPOCHS = 30

LR = 5e-4
WEIGHT_DECAY = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
SEED = 42
set_seed(SEED)

# Dataset

In [4]:
# ===============================
# Load CSVs
# ===============================
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train samples:", len(train_df))
print("Val samples  :", len(val_df))
print("Test samples :", len(test_df))

# ===============================
# Check required columns
# ===============================
required_cols = [
    "new_file_path",
    "label_text",
    "three_class_label",
    "binary_label",
    "patient_id"
]

for col in required_cols:
    assert col in train_df.columns, f"Missing column in train_df: {col}"
    assert col in val_df.columns,  f"Missing column in val_df: {col}"
    assert col in test_df.columns,  f"Missing column in test_df: {col}"

print("✅ All required columns present")

# ===============================
# Class distribution
# ===============================
print("\nTrain distribution:")
print(train_df["label_text"].value_counts())

print("\nValidation distribution:")
print(val_df["label_text"].value_counts())

print("\nTest distribution:")
print(test_df["label_text"].value_counts())

# ===============================
# Peek at data
# ===============================
train_df.head()


Train samples: 1705
Val samples  : 656
Test samples : 646
✅ All required columns present

Train distribution:
label_text
Normal    962
AMD       413
DME       330
Name: count, dtype: int64

Validation distribution:
label_text
Normal    290
AMD       225
DME       141
Name: count, dtype: int64

Test distribution:
label_text
Normal    333
AMD       219
DME        94
Name: count, dtype: int64


,label_text,patient_id,new_file_path,three_class_label,binary_label,fold
0,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0
1,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0
2,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0
3,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0
4,Normal,NORMAL_1,/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD...,0,0,0


In [5]:
# # import pandas as pd
# from sklearn.model_selection import train_test_split

# # 1. Identify unique patients and their corresponding labels
# # We use the 'first' or 'mode' label for each patient to ensure stratification


# # 2. Load Dataset
# file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ucsd_filtered.csv'
# df = pd.read_csv(file_path)


# patient_data = df.groupby('patient_id')['binary_label'].agg(lambda x: x.mode()[0]).reset_index()

# # # 2. Split Patients: 10% (Train+Val) and 90% (Test)
# # train_val_patients, test_patients = train_test_split(
# #     patient_data, 
# #     test_size=0.90, 
# #     stratify=patient_data['binary_label'], 
# #     random_state=SEED
# # )

# train_val_patients, test_patients = train_test_split(
#     patient_data, 
#     test_size=0.50, 
#     stratify=patient_data['binary_label'], 
#     random_state=SEED
# )

# # 2. Split Patients: 15% (Train+Val) and 95% (Test)
# # train_val_patients, test_patients = train_test_split(
# #     patient_data, 
# #     test_size=0.95, 
# #     stratify=patient_data['binary_label'], 
# #     random_state=SEED
# # )

# # 3. Split the 10% Patient chunk into 8% Train and 2% Val (0.02/0.10 = 0.20)
# train_patients, val_patients = train_test_split(
#     train_val_patients, 
#     test_size=0.20, 
#     stratify=train_val_patients['binary_label'], 
#     random_state=SEED
# )

# # 4. Map the split patients back to the original full dataframe
# train_df = df[df['patient_id'].isin(train_patients['patient_id'])].copy()
# val_df   = df[df['patient_id'].isin(val_patients['patient_id'])].copy()
# test_df  = df[df['patient_id'].isin(test_patients['patient_id'])].copy()

# # --- Verification ---
# print(f"Total Patients: {len(patient_data)}")
# print(f"Train Patients: {len(train_patients)} | Val Patients: {len(val_patients)} | Test Patients: {len(test_patients)}")
# print("-" * 30)
# print(f"Total Samples: {len(df)}")
# print(f"Train Samples: {len(train_df)} ({len(train_df)/len(df):.1%})")
# print(f"Val Samples:   {len(val_df)} ({len(val_df)/len(df):.1%})")
# print(f"Test Samples:  {len(test_df)} ({len(test_df)/len(df):.1%})")

# # Check for leakage (should be 0)
# overlap = set(train_df['patient_id']).intersection(set(test_df['patient_id']))
# print(f"\nPatient ID Overlap between Train and Test: {len(overlap)}")

In [6]:
# file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv'
# import pandas as pd
# df = pd.read_csv(file_path)
# # 1. Identify unique patients and their specific category (AMD, DME, or NORMAL)
# # We use the first part of the patient_id string to identify the type
# patient_data = df.groupby('patient_id').first().reset_index()
# patient_data['category'] = patient_data['patient_id'].str.extract(r'([A-Za-z]+)')

# # Define the pool of patients
# all_patients = patient_data.copy()

# # 10 percent 

# # # 2. SELECT VALIDATION PATIENTS (1 Normal, 1 from AMD/DME pool)
# # val_normal = all_patients[all_patients['category'] == 'NORMAL'].sample(1, random_state=SEED)
# # val_abnormal = all_patients[all_patients['category'].isin(['AMD'])].sample(1, random_state=SEED)
# # val_dme    = all_patients[all_patients['category'] == 'DME'].sample(1, random_state=42)

# # val_list = pd.concat([val_normal, val_abnormal, val_dme])

# # # Remove validation from pool
# # pool_after_val = all_patients[~all_patients['patient_id'].isin(val_list['patient_id'])]

# # # 3. SELECT TRAINING PATIENTS (1 Normal, 1 AMD, 1 DME)
# # # Note: Using seed 42 as requested
# # train_normal = pool_after_val[pool_after_val['category'] == 'NORMAL'].sample(1, random_state=42)
# # train_amd    = pool_after_val[pool_after_val['category'] == 'AMD'].sample(1, random_state=42)
# # train_dme    = pool_after_val[pool_after_val['category'] == 'DME'].sample(1, random_state=42)
# # train_list   = pd.concat([train_normal, train_amd, train_dme])

# # 50 percent

# # 2. SELECT VALIDATION PATIENTS (1 Normal, 1 from AMD/DME pool)
# val_normal = all_patients[all_patients['category'] == 'NORMAL'].sample(2, random_state=SEED)
# val_abnormal = all_patients[all_patients['category'].isin(['AMD'])].sample(1, random_state=SEED)
# val_dme    = all_patients[all_patients['category'] == 'DME'].sample(1, random_state=42)

# val_list = pd.concat([val_normal, val_abnormal, val_dme])

# # Remove validation from pool
# pool_after_val = all_patients[~all_patients['patient_id'].isin(val_list['patient_id'])]

# # 3. SELECT TRAINING PATIENTS (1 Normal, 1 AMD, 1 DME)
# # Note: Using seed 42 as requested
# train_normal = pool_after_val[pool_after_val['category'] == 'NORMAL'].sample(6, random_state=42)
# train_amd    = pool_after_val[pool_after_val['category'] == 'AMD'].sample(4, random_state=42)
# train_dme    = pool_after_val[pool_after_val['category'] == 'DME'].sample(5, random_state=42)
# train_list   = pd.concat([train_normal, train_amd, train_dme])



# # 4. REMAINING ARE TEST
# test_list = pool_after_val[~pool_after_val['patient_id'].isin(train_list['patient_id'])]

# # 5. MAP BACK TO DATASET
# train_df = df[df['patient_id'].isin(train_list['patient_id'])].copy()
# val_df   = df[df['patient_id'].isin(val_list['patient_id'])].copy()
# test_df  = df[df['patient_id'].isin(test_list['patient_id'])].copy()

# # --- VERIFICATION ---
# print(f"Train Patients ({len(train_list)}): {train_list['patient_id'].tolist()}")
# print(f"Val Patients   ({len(val_list)}): {val_list['patient_id'].tolist()}")
# print(f"Test Patients  ({len(test_list)})")
# print("-" * 30)
# print(f"Image Counts -> Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# print(f"Total Samples: {len(df)}")
# print(f"Train Samples: {len(train_df)} ({len(train_df)/len(df):.1%})")
# print(f"Val Samples:   {len(val_df)} ({len(val_df)/len(df):.1%})")
# print(f"Test Samples:  {len(test_df)} ({len(test_df)/len(df):.1%})")


In [7]:
# import pandas as pd
# from sklearn.model_selection import train_test_split
# import numpy as np
# import random
# import torch

# # 1. Set Seed for Reproducibility
# def set_seed(seed=42):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(seed)
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False

# SEED = 42
# set_seed(SEED)

# # 2. Load Dataset
# file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/octC8_filtered.csv'
# df = pd.read_csv(file_path)

# # 3. Split the entire data into 10% (for train+val) and 90% (for test)

# # # This ensures 90% of the total data goes to test_df
# # train_val_df, test_df = train_test_split(
# #     df, 
# #     test_size=0.90, 
# #     stratify=df['binary_label'], 
# #     random_state=SEED
# # )
# train_val_df, test_df = train_test_split(
#     df, 
#     test_size=0.50, 
#     stratify=df['binary_label'], 
#     random_state=SEED
# )


# # This ensures 95% of the total data goes to test_df
# # train_val_df, test_df = train_test_split(
# #     df, 
# #     test_size=0.95, 
# #     stratify=df['binary_label'], 
# #     random_state=SEED
# # )

# # 4. Split the 10% chunk into 8% Train and 2% Val
# # Note: 2% of total is 20% of this 10% chunk (0.02 / 0.10 = 0.20)
# train_df, val_df = train_test_split(
#     train_val_df, 
#     test_size=0.20, 
#     stratify=train_val_df['binary_label'], 
#     random_state=SEED
# )

# # Verification
# print(f"Total samples: {len(df)}")
# print(f"Train size (8%): {len(train_df)}")
# print(f"Val size   (2%): {len(val_df)}")
# print(f"Test size (90%): {len(test_df)}")

# # Check label distribution
# print("\nLabel Distribution (should be ~52.7% label 1):")
# print(f"Train: \n{train_df['binary_label'].value_counts()}")
# print(f"Val: \n{val_df['binary_label'].value_counts()}")

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the data
file_path = '/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/dhu_filtered.csv'
df = pd.read_csv(file_path)

# 2. First Split: Separate the 95% test set from the 5% "remainder"
# Stratify ensures both sets have the same % of positive/negative cases
df_temp, test_df = train_test_split(
    df, 
    test_size=0.95, 
    stratify=df['binary_label'], 
    random_state=42
)

# 3. Second Split: Split the 5% remainder into Train (4%) and Valid (1%)
# Since 1% is 1/5 of the temp set, we use test_size=0.2 (20% of 5% = 1%)
train_df, val_df = train_test_split(
    df_temp, 
    test_size=0.20, 
    stratify=df_temp['binary_label'], 
    random_state=42
)

# 4. Print the distributions
def print_dist(name, d):
    counts = d['binary_label'].value_counts()
    percents = d['binary_label'].value_counts(normalize=True) * 100
    print(f"--- {name} ({len(d)} samples) ---")
    for val in counts.index:
        print(f"Label {val}: {counts[val]} ({percents[val]:.2f}%)")
    print()

print_dist("Train Set (4%)", train_df)
print_dist("Valid Set (1%)", val_df)
print_dist("Test Set (95%)", test_df)

--- Train Set (4%) (114 samples) ---
Label 1: 58 (50.88%)
Label 0: 56 (49.12%)

--- Valid Set (1%) (29 samples) ---
Label 1: 15 (51.72%)
Label 0: 14 (48.28%)

--- Test Set (95%) (2722 samples) ---
Label 1: 1385 (50.88%)
Label 0: 1337 (49.12%)



In [9]:
from PIL import Image, ImageFile
import torch

ImageFile.LOAD_TRUNCATED_IMAGES = True

class NEHOCTDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform=None, label_col="binary_label"):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.label_col = label_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            img = Image.open(row["new_file_path"])

            # Ensure 3-channel RGB
            if img.mode != "RGB":
                img = img.convert("RGB")

        except Exception as e:
            raise RuntimeError(
                f"Failed to load image: {row['new_file_path']}"
            ) from e

        label = int(row[self.label_col])

        if self.transform:
            img = self.transform(img)

        return img, label


In [10]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [11]:
# import os
# TEST_CSV = "4096_3328_SD_imbalanced_val_15.csv"
# ROOT = "/media/miglab/DATA_20TB1/Datasets/EMBED"
# df = pd.read_csv(TEST_CSV)
# df["new_file_path"] = df["anon_dicom_path"].apply(
#     lambda x: os.path.join(ROOT, x) if pd.notna(x) else x
# )
# df.to_csv(TEST_CSV, index=False)

In [12]:
train_dataset = NEHOCTDataset(train_df, transform=train_transform, label_col='binary_label')
val_dataset   = NEHOCTDataset(val_df,   transform=test_transform, label_col='binary_label')
test_dataset  = NEHOCTDataset(test_df,  transform=test_transform, label_col='binary_label')

pinmem = True if torch.cuda.is_available() else False

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=pinmem)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pinmem)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pinmem)

In [13]:
images, _ = next(iter(train_loader))
print(images.min(), images.max(), images.mean())
print(images.shape)  # (B, C, H, W)

tensor(-2.1179) tensor(2.6400) tensor(-1.1305)
torch.Size([32, 3, 512, 512])


In [14]:
# import matplotlib.pyplot as plt

# IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
# IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# def denormalize(img):
#     """
#     img: Tensor [3, H, W] after Normalize
#     """
#     return img * IMAGENET_STD + IMAGENET_MEAN
# def visualize_dataloader_batch(dataloader, num_images=4, title="After DataLoader"):
#     """
#     Visualize images exactly as seen by the model
#     """
#     images, labels = next(iter(dataloader))

#     num_images = min(num_images, images.size(0))

#     plt.figure(figsize=(4 * num_images, 4))
#     for i in range(num_images):
#         img = images[i].cpu()
#         img = denormalize(img)
#         img = img.clamp(0, 1)

#         plt.subplot(1, num_images, i + 1)
#         plt.imshow(img.permute(1, 2, 0))
#         plt.title(f"Label: {labels[i].item()}")
#         plt.axis("off")

#     plt.suptitle(title, fontsize=14)
#     plt.show()

# visualize_dataloader_batch(train_loader, num_images=8, title="Train Loader Output")
# visualize_dataloader_batch(val_loader,   num_images=8, title="Val Loader Output")
# visualize_dataloader_batch(test_loader,  num_images=8, title="Test Loader Output")

# Metrics

In [15]:
def compute_uncertainty_stats(y_proba: np.ndarray, y_true: np.ndarray):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    y_true  = np.asarray(y_true, dtype=np.int64)

    y_proba = np.nan_to_num(y_proba, nan=0.5, posinf=1.0, neginf=0.0)

    eps = 1e-8
    p1 = np.clip(y_proba, eps, 1.0 - eps)
    p0 = 1.0 - p1

    entropy = -(p0 * np.log(p0) + p1 * np.log(p1))
    confidence = np.maximum(p0, p1)
    uncertainty = 1.0 - confidence

    out = {
        "avg_entropy": float(np.mean(entropy)),
        "entropy_std": float(np.std(entropy)),
        "avg_uncertainty": float(np.mean(uncertainty)),
        "uncertainty_std": float(np.std(uncertainty)),
    }

    # ---- class-conditional statistics ----
    for cls in (0, 1):
        mask = (y_true == cls)

        if mask.any():
            out[f"entropy_class{cls}_avg"] = float(entropy[mask].mean())
            out[f"entropy_class{cls}_std"] = float(entropy[mask].std())
            out[f"uncertainty_class{cls}_avg"] = float(uncertainty[mask].mean())
            out[f"uncertainty_class{cls}_std"] = float(uncertainty[mask].std())
        else:
            out[f"entropy_class{cls}_avg"] = float("nan")
            out[f"entropy_class{cls}_std"] = float("nan")
            out[f"uncertainty_class{cls}_avg"] = float("nan")
            out[f"uncertainty_class{cls}_std"] = float("nan")

    return out

In [16]:
def compute_binary_class_weights(df):
    counts = df["binary_label"].value_counts().sort_index().values
    total = counts.sum()
    weights = total / (2.0 * counts)
    return torch.tensor(weights, dtype=torch.float)

class_weights = compute_binary_class_weights(train_df).to(DEVICE)

print("Class counts:")
print(train_df["binary_label"].value_counts().sort_index())

print("Class weights:", class_weights.cpu().numpy())

Class counts:
binary_label
0    56
1    58
Name: count, dtype: int64
Class weights: [1.0178572  0.98275864]


In [17]:
# import csv
# import os

# CSV_HEADER = [
#     "epoch",
#     "train_loss", "val_loss", "test_loss",
#     "train_acc", "val_acc", "test_acc",
#     "train_auc", "val_auc", "test_auc",
#     "val_pr_auc", "test_pr_auc",
#     "val_f1", "test_f1", "val_macro_f1", "test_macro_f1",
#     "val_precision", "val_recall", "val_npv",
#     "test_precision", "test_recall", "test_npv",
#     "val_specificity", "test_specificity",
#     "val_sens_at_spec_90", "test_sens_at_spec_90",
#     "avg_entropy", "entropy_std",
#     "avg_uncertainty", "uncertainty_std",
#     "entropy_class0_avg", "entropy_class0_std",
#     "entropy_class1_avg", "entropy_class1_std",
#     "uncertainty_class0_avg", "uncertainty_class0_std",
#     "uncertainty_class1_avg", "uncertainty_class1_std",
#     "tn", "fp", "fn", "tp", "n_samples"
# ]

# def append_metrics_to_csv(csv_path, row_dict, float_fmt="{:.6f}"):
#     """
#     Appends one epoch of metrics to CSV.
#     Floats are formatted to fixed precision (default .6f).
#     Creates file + header if missing.
#     """
#     file_exists = os.path.isfile(csv_path)

#     formatted_row = {}
#     for k, v in row_dict.items():
#         if isinstance(v, float):
#             if np.isnan(v):
#                 formatted_row[k] = np.nan
#             else:
#                 formatted_row[k] = float_fmt.format(v)
#         else:
#             formatted_row[k] = v

#     with open(csv_path, mode="a", newline="") as f:
#         writer = csv.DictWriter(f, fieldnames=CSV_HEADER)

#         if not file_exists:
#             writer.writeheader()

#         writer.writerow({k: formatted_row.get(k, np.nan) for k in CSV_HEADER})

In [18]:
import csv
import os

CSV_HEADER = [
    "epoch",
    "train_loss", "val_loss",
    "train_acc", "val_acc", 
    "train_auc", "val_auc",
    "val_pr_auc", 
    "val_f1", "val_macro_f1", 
    "val_precision", "val_recall", "val_npv",
    "val_specificity", 
    "val_sens_at_spec_90", 
    # "avg_entropy", "entropy_std",
    # "avg_uncertainty", "uncertainty_std",
    # "entropy_class0_avg", "entropy_class0_std",
    # "entropy_class1_avg", "entropy_class1_std",
    # "uncertainty_class0_avg", "uncertainty_class0_std",
    # "uncertainty_class1_avg", "uncertainty_class1_std",
    # "tn", "fp", "fn", "tp", "n_samples"
]

def append_metrics_to_csv(csv_path, row_dict, float_fmt="{:.6f}"):
    """
    Appends one epoch of metrics to CSV.
    Floats are formatted to fixed precision (default .6f).
    Creates file + header if missing.
    """
    file_exists = os.path.isfile(csv_path)

    formatted_row = {}
    for k, v in row_dict.items():
        if isinstance(v, float):
            if np.isnan(v):
                formatted_row[k] = np.nan
            else:
                formatted_row[k] = float_fmt.format(v)
        else:
            formatted_row[k] = v

    with open(csv_path, mode="a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_HEADER)

        if not file_exists:
            writer.writeheader()

        writer.writerow({k: formatted_row.get(k, np.nan) for k in CSV_HEADER})

In [19]:
from sklearn.metrics import fbeta_score

def set_size_and_singleton(y_proba, tau=0.9):
    """
    Binary prediction sets based on confidence threshold tau.
    tau ∈ (0.5, 1.0)
    """
    y_proba = np.asarray(y_proba, dtype=np.float64)

    confidence = np.maximum(y_proba, 1.0 - y_proba)

    singleton_mask = confidence >= tau          # confident → singleton
    set_sizes = np.where(singleton_mask, 1, 2)  # else abstain

    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)),
    }

# def expected_calibration_error(y_true, y_proba, n_bins=10):
#     y_true = np.asarray(y_true, dtype=int)
#     y_proba = np.asarray(y_proba, dtype=np.float64)

#     # predicted class and confidence
#     y_pred = (y_proba >= 0.5).astype(int)
#     conf   = np.where(y_pred == 1, y_proba, 1.0 - y_proba)

#     bins = np.linspace(0.0, 1.0, n_bins + 1)
#     bin_ids = np.digitize(conf, bins) - 1

#     ece = 0.0
#     for b in range(n_bins):
#         mask = bin_ids == b
#         if not np.any(mask):
#             continue

#         acc_b  = np.mean(y_pred[mask] == y_true[mask])
#         conf_b = np.mean(conf[mask])
#         ece   += np.abs(acc_b - conf_b) * np.mean(mask)

#     return float(ece)

def expected_calibration_error(y_true, y_proba, n_bins=15):
    """
    Standard ECE for binary classification (Guo et al., 2017)
    """
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    # Prediction and confidence
    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.maximum(y_proba, 1.0 - y_proba)

    # Bin edges: confidence ∈ [0.5, 1.0]
    bin_edges = np.linspace(0.5, 1.0, n_bins + 1)

    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        bin_lower = bin_edges[i]
        bin_upper = bin_edges[i + 1]

        mask = (conf > bin_lower) & (conf <= bin_upper)
        if not np.any(mask):
            continue

        acc_bin = np.mean(y_pred[mask] == y_true[mask])
        conf_bin = np.mean(conf[mask])
        ece += (np.sum(mask) / n) * abs(acc_bin - conf_bin)

    return float(ece)

In [20]:
def compute_binary_metrics(y_proba, y_true, threshold=0.5, target_spec=0.90):
    """
    y_proba: list or np.array of predicted probability for class 1
    y_true:  list or np.array of integer {0,1}
    threshold: float for converting proba -> label
    returns: dict of metrics including confusion matrix entries
    """
    y_proba = np.asarray(y_proba)
    y_true = np.asarray(y_true).astype(int)

    out = {}
    # ROC-AUC and PR-AUC (handle single-class edge)
    try:
        out['roc_auc'] = float(roc_auc_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan")
    except Exception:
        out['roc_auc'] = float("nan")
    try:
        out['pr_auc'] = float(average_precision_score(y_true, y_proba)) if len(np.unique(y_true)) > 1 else float("nan")
    except Exception:
        out['pr_auc'] = float("nan")

    # Binary predictions
    y_pred = (y_proba >= threshold).astype(int)

    # Standard metrics
    out['precision'] = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
    out['recall'] = float(recall_score(y_true, y_pred, average="macro", zero_division=0))    # sensitivity
    out['f1'] = float(f1_score(y_true, y_pred, zero_division=0))
    out['macro_f1'] = float(
        f1_score(y_true, y_pred, average="macro", zero_division=0)
    )
    out['balanced_acc'] = float(balanced_accuracy_score(y_true, y_pred))
    # MCC (may raise if degenerate)
    # try:
    #     out['mcc'] = float(matthews_corrcoef(y_true, y_pred))
    # except Exception:
    #     out['mcc'] = float("nan")

    # Confusion matrix
    try:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
        out['tn'] = int(tn); out['fp'] = int(fp); out['fn'] = int(fn); out['tp'] = int(tp)
        out['specificity'] = float(tn / (tn + fp)) if (tn + fp) > 0 else float("nan")
        out['sensitivity'] = float(tp / (tp + fn)) if (tp + fn) > 0 else float("nan")
        out['npv'] = float(tn / (tn + fn)) if (tn + fn) > 0 else float("nan")

    except Exception:
        out['tn'] = out['fp'] = out['fn'] = out['tp'] = 0
        out['specificity'] = out['sensitivity'] = float("nan")

    # Brier score
    try:
        out['brier'] = float(brier_score_loss(y_true, y_proba))
    except Exception:
        out['brier'] = float("nan")

    out['n_samples'] = int(len(y_true))
    out['threshold'] = float(threshold)
    try:
        thresh_list = np.linspace(0, 1, 200)
        best_sens = float("nan")
        best_thr = float("nan")

        for thr in thresh_list:
            yp = (y_proba >= thr).astype(int)
            cm = confusion_matrix(y_true, yp, labels=[0,1])
            tn, fp, fn, tp = cm.ravel()
            spec = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
            sens = tp / (tp + fn) if (tp + fn) > 0 else float("nan")

            if not np.isnan(spec) and spec >= target_spec:
                # choose largest sensitivity among valid thresholds
                if np.isnan(best_sens) or sens > best_sens:
                    best_sens = sens
                    best_thr = thr

        out["sens_at_spec"] = best_sens
        out["thr_at_spec"] = best_thr
        out["target_spec"] = target_spec

    except Exception:
        out["sens_at_spec"] = float("nan")
        out["thr_at_spec"] = float("nan")
        out["target_spec"] = target_spec
    
    try:
        u_stats = compute_uncertainty_stats(y_proba, y_true)
        out.update(u_stats)
    except Exception:
        for k in [
            "avg_entropy","entropy_std",
            "avg_uncertainty","uncertainty_std",
            "entropy_class0_avg","entropy_class0_std",
            "entropy_class1_avg","entropy_class1_std",
            "uncertainty_class0_avg","uncertainty_class0_std",
            "uncertainty_class1_avg","uncertainty_class1_std",
        ]:
            out[k] = float("nan")
    return out

# def print_epoch_summary(epoch, train_loss, val_loss, test_loss, train_auc, val_auc, test_auc, train_acc, val_acc, test_acc, metrics, uncert):
#     print("=" * 90)
#     print(f"Epoch {epoch}")
#     print("-" * 90)
#     print(f"Train | Loss={(train_loss):.4f}  AUC={(train_auc):.4f}  Acc={(train_acc):.4f}")
#     print(f"Val   | Loss={(val_loss):.4f}  AUC={(val_auc):.4f}  Acc={(val_acc):.4f}")
#     print(f"Test  | Loss={(test_loss):.4f}  AUC={(test_auc):.4f}  Acc={(test_acc):.4f}")

#     print("\nClassification Metrics:")
#     print(
#         f"  ROC-AUC={(metrics['roc_auc']):.4f}  PR-AUC={(metrics['pr_auc']):.4f}  Brier_score={(metrics['brier']):.4f}  "
#         f"F1={(metrics['f1']):.4f}  Macro-F1={(metrics['macro_f1']):.4f}  Precision={(metrics['precision']):.4f}  Recall={(metrics['recall']):.4f}  NPV={(metrics['npv']):.4f}" f"  Specificity={(metrics['specificity']):.4f}  "  f"Sens@Spec90={(metrics['sens_at_spec']):.4f} ")

#     print("\nUncertainty Metrics:")
#     print(
#         f"  AvgEntropy={(uncert['avg_entropy']):.4f} ± {(uncert['entropy_std']):.4f} | " f"AvgUncertainty={(uncert['avg_uncertainty']):.4f} ± {(uncert['uncertainty_std']):.4f}" f"  Entropy(C0)={(uncert['entropy_class0_avg']):.4f}  " f"Entropy(C1)={(uncert['entropy_class1_avg']):.4f}" f"  Uncertainty(C0)={(uncert['uncertainty_class0_avg']):.4f}  " f"Uncertainty(C1)={(uncert['uncertainty_class1_avg']):.4f}"
#     )

#     print("\nConfusion Matrix:" f"  TP={metrics['tp']}  FP={metrics['fp']} " f"FN={metrics['fn']}  TN={metrics['tn']}")
#     print("=" * 90)


def print_epoch_summary(epoch, train_loss, val_loss, test_loss, train_auc, val_auc, test_auc, train_acc, val_acc, test_acc, metrics):
    print("=" * 90)
    print(f"Epoch {epoch}")
    print("-" * 90)
    print(f"Train | Loss={(train_loss):.4f}  AUC={(train_auc):.4f}  Acc={(train_acc):.4f}")
    print(f"Val   | Loss={(val_loss):.4f}  AUC={(val_auc):.4f}  Acc={(val_acc):.4f}")
    print(f"Test  | Loss={(test_loss):.4f}  AUC={(test_auc):.4f}  Acc={(test_acc):.4f}")

    print("\nClassification Metrics:")
    print(
        f"  ROC-AUC={(metrics['roc_auc']):.4f}  PR-AUC={(metrics['pr_auc']):.4f}  Brier_score={(metrics['brier']):.4f}  "
        f"F1={(metrics['f1']):.4f}  Macro-F1={(metrics['macro_f1']):.4f}  Precision={(metrics['precision']):.4f}  Recall={(metrics['recall']):.4f}  NPV={(metrics['npv']):.4f}" f"  Specificity={(metrics['specificity']):.4f}  "  f"Sens@Spec90={(metrics['sens_at_spec']):.4f} ")

    # print("\nUncertainty Metrics:")
    # print(
    #     f"  AvgEntropy={(uncert['avg_entropy']):.4f} ± {(uncert['entropy_std']):.4f} | " f"AvgUncertainty={(uncert['avg_uncertainty']):.4f} ± {(uncert['uncertainty_std']):.4f}" f"  Entropy(C0)={(uncert['entropy_class0_avg']):.4f}  " f"Entropy(C1)={(uncert['entropy_class1_avg']):.4f}" f"  Uncertainty(C0)={(uncert['uncertainty_class0_avg']):.4f}  " f"Uncertainty(C1)={(uncert['uncertainty_class1_avg']):.4f}"
    # )

    print("\nConfusion Matrix:" f"  TP={metrics['tp']}  FP={metrics['fp']} " f"FN={metrics['fn']}  TN={metrics['tn']}")

# Model

In [21]:
def get_resnet34_binary(pretrained=True, num_classes=2):
    model = models.resnet34(pretrained=pretrained)
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes) 
    return model

model = get_resnet34_binary(pretrained=True, num_classes=NUM_CLASSES)
model = model.to(DEVICE)

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [22]:
def dirichlet_kl(alpha, num_classes):
    # KL between Dir(alpha) and Dir(1) (uniform prior)
    # alpha: [B, C]
    S = torch.sum(alpha, dim=1, keepdim=True)  # [B,1]
    ones = torch.ones_like(alpha)
    # lgamma sums
    t1 = torch.lgamma(S) - torch.sum(torch.lgamma(alpha), dim=1, keepdim=True)
    # since beta = ones => terms with lgamma(1)=0 drop
    t2 = torch.sum((alpha - ones) * (torch.digamma(alpha) - torch.digamma(S)), dim=1, keepdim=True)
    kl = t1 + t2
    return kl  # [B,1]

def edl_loss_from_logits(logits, y_onehot, epoch, num_classes=2):
    """
    logits: [B,C] raw outputs (before softmax)
    y_onehot: [B,C] float
    epoch: current epoch (1-indexed); used for KL annealing as min(1, epoch/50)
    returns: loss_scalar, (err_mean, var_mean, kl_mean), extra dict for logging
    """
    # Official code uses ReLU for evidence
    # evidence = torch.relu(logits)            # [B,C]
    evidence = F.softplus(logits)        # [B,C]
    alpha = evidence + 1.0                   # [B,C]
    # alpha = evidence + 1

    S = torch.sum(alpha, dim=1, keepdim=True)  # [B,1]
    # predictive mean
    p_hat = alpha / S                        # [B,C]
    # (a) error term
    err = torch.sum((y_onehot - p_hat) ** 2, dim=1, keepdim=True)  # [B,1]
    # (b) variance term
    var = torch.sum(alpha * (S - alpha) / (S*S*(S + 1.0)), dim=1, keepdim=True)  # [B,1]
    # data term per sample
    data_loss = err + var   # [B,1]
    # (c) KL term with the stabilized alpha_tilde (as official code)
    # alpha_tilde = (1 - y_true) * alpha + y_true
    alpha_tilde = (1.0 - y_onehot) * alpha + y_onehot
    kl = dirichlet_kl(alpha_tilde, num_classes)  # [B,1]
    # annealing coefficient: min(1.0, epoch/10)
    # anneal_coeff = min(1.0, float(epoch) / 10.0)
    anneal_coeff = min(1.0, float(epoch) / 50.0)
    # per-sample loss: sum data terms + anneal * kl
    per_sample = data_loss + anneal_coeff * kl  # [B,1]
    loss = torch.mean(per_sample)  # scalar
    # return helpful components
    err_mean = torch.mean(err).item()
    var_mean = torch.mean(var).item()
    kl_mean = torch.mean(kl).item()
    evidence_mean = torch.mean(torch.sum(evidence, dim=1)).item()  # mean total evidence per sample
    
    extras = {
        "p_hat": p_hat.detach(),
        "evidence": evidence.detach(),
        "alpha": alpha.detach(),
        "S": S.detach()
    }
    return loss, (err_mean, var_mean, kl_mean, evidence_mean), extras

optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

# Loops

In [23]:
def set_bn_eval(m):
    if isinstance(m, nn.BatchNorm2d):
        m.eval()

def train_one_epoch_edl(model, loader, optimizer, epoch):
    model.train()
    model.apply(set_bn_eval)
    running_loss = 0.0
    preds = []
    targets_all = []
    evidences = []   # total evidence per sample
    correct_flags = []

    count = 0
    all_uncertainty_stats = []
    for imgs, targets in tqdm(loader, desc=f"Train Epoch {epoch}", leave=False):
        # count += 1
        # if count == 5:
        #     break
        imgs = imgs.to(DEVICE)
        targets = targets.to(DEVICE)
        optimizer.zero_grad()

        logits = model(imgs)  # [B,C]
        
        # one-hot labels
        # y_onehot = torch.zeros(logits.size(), device=DEVICE)
        # y_onehot.scatter_(1, targets.unsqueeze(1), 1.0)
        y_onehot = F.one_hot(targets, num_classes=NUM_CLASSES).float().to(device=DEVICE)
        
        loss, (err_m, var_m, kl_m, ev_m), extras = edl_loss_from_logits(logits, y_onehot, epoch, num_classes=NUM_CLASSES)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
        # predictions from predictive mean p_hat
        p_hat = extras["p_hat"].cpu().numpy()  # [B,C]
        ev = extras["evidence"].sum(dim=1).cpu().numpy()  # total evidence per sample [B]
        
        # convert to scalar predicted probability for class 1 (for binary)
        pred_probs_class1 = p_hat[:,1] if NUM_CLASSES>1 else p_hat[:,0]
        preds.extend(pred_probs_class1.tolist())
        evidences.extend(ev.tolist())
        targets_all.extend(targets.cpu().numpy().tolist())
        # correctness flags
        preds_label = np.argmax(p_hat, axis=1)
        correct_flags.extend((preds_label == targets.cpu().numpy()).tolist())

        stats = compute_uncertainty_stats(pred_probs_class1, targets.cpu().numpy())
        all_uncertainty_stats.append(stats)

    epoch_loss = running_loss / len(loader.dataset)
    try:
        auc = roc_auc_score(targets_all, preds) if len(set(targets_all))>1 else float("nan")
    except Exception:
        auc = float("nan")
    acc = accuracy_score(targets_all, [1 if p>=0.5 else 0 for p in preds]) 

    return epoch_loss, auc, acc, evidences, targets_all, correct_flags, preds, stats, all_uncertainty_stats

In [24]:
def validate_edl(model, loader, epoch):
    model.eval()
    running_loss = 0.0
    preds = []
    targets_all = []
    evidences = []
    correct_flags = []
    all_uncertainty_stats = []

    cout = 0
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Val", leave=False):
            # cout += 1
            # if cout == 5:
            #     break
            imgs = imgs.to(DEVICE)
            targets = targets.to(DEVICE)

            logits = model(imgs)
            
            y_onehot = F.one_hot(targets, num_classes=NUM_CLASSES).float()

            loss, (err_m, var_m, kl_m, ev_m), extras = edl_loss_from_logits(logits, y_onehot, epoch, num_classes=NUM_CLASSES)
            running_loss += loss.item() * imgs.size(0)
            p_hat = extras["p_hat"].cpu().numpy()
            ev = extras["evidence"].sum(dim=1).cpu().numpy()
            pred_probs_class1 = p_hat[:,1] if NUM_CLASSES>1 else p_hat[:,0]
            preds.extend(pred_probs_class1.tolist())
            evidences.extend(ev.tolist())
            targets_all.extend(targets.cpu().numpy().tolist())
            preds_label = np.argmax(p_hat, axis=1)
            correct_flags.extend((preds_label == targets.cpu().numpy()).tolist())

            # NEW: uncertainties
            stats = compute_uncertainty_stats(pred_probs_class1, targets.cpu().numpy())
            all_uncertainty_stats.append(stats)

    epoch_loss = running_loss / len(loader.dataset)
    try:
        auc = roc_auc_score(targets_all, preds) if len(set(targets_all))>1 else float("nan")
    except Exception:
        auc = float("nan")
    acc = accuracy_score(targets_all, [1 if p>=0.5 else 0 for p in preds])
    metrics = compute_binary_metrics(preds, targets_all)

    return epoch_loss, auc, acc, evidences, targets_all, correct_flags, preds, stats, all_uncertainty_stats, metrics

In [25]:
def test_edl(model, loader, epoch):
    model.eval()
    running_loss = 0.0
    preds = []
    targets_all = []
    evidences = []
    correct_flags = []
    all_uncertainty_stats = []

    cout = 0
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Test", leave=False):
            # cout += 1
            # if cout == 5:
            #     break
            imgs = imgs.to(DEVICE)
            targets = targets.to(DEVICE)

            logits = model(imgs)
            
            y_onehot = F.one_hot(targets, num_classes=NUM_CLASSES).float()

            loss, (err_m, var_m, kl_m, ev_m), extras = edl_loss_from_logits(logits, y_onehot, epoch, num_classes=NUM_CLASSES)
            running_loss += loss.item() * imgs.size(0)
            p_hat = extras["p_hat"].cpu().numpy()
            ev = extras["evidence"].sum(dim=1).cpu().numpy()
            pred_probs_class1 = p_hat[:,1] if NUM_CLASSES>1 else p_hat[:,0]
            preds.extend(pred_probs_class1.tolist())
            evidences.extend(ev.tolist())
            targets_all.extend(targets.cpu().numpy().tolist())
            preds_label = np.argmax(p_hat, axis=1)
            correct_flags.extend((preds_label == targets.cpu().numpy()).tolist())

            # NEW: uncertainties
            stats = compute_uncertainty_stats(pred_probs_class1, targets.cpu().numpy())
            all_uncertainty_stats.append(stats)
            
    epoch_loss = running_loss / len(loader.dataset)
    try:
        auc = roc_auc_score(targets_all, preds) if len(set(targets_all))>1 else float("nan")
    except Exception:
        auc = float("nan")
    acc = accuracy_score(targets_all, [1 if p>=0.5 else 0 for p in preds])
    metrics = compute_binary_metrics(preds, targets_all)

    return epoch_loss, auc, acc, evidences, targets_all, correct_flags, preds, stats, all_uncertainty_stats, metrics

In [26]:
def aggregate_uncertainty_stats(stats_list):
    if len(stats_list) == 0:
        return {}

    keys = stats_list[0].keys()
    out = {}

    for k in keys:
        vals = [d[k] for d in stats_list]
        out[k] = float(np.mean(vals))

    return out

train_unc_history = []
val_unc_history   = []
test_unc_history = []
train_ev_history  = [] 
val_ev_history    = []
test_ev_history = []
L_train_acc = []
L_train_ev_s = []   # mean evidence on successful preds
L_train_ev_f = []   # mean evidence on failed preds
L_val_acc = []
L_val_ev_s = []
L_val_ev_f = []
L_test_acc = []
L_test_ev_s = []
L_test_ev_f = []

# Training  

In [27]:
# import pandas as pd

# models_path = "ResNet34_neh_reduced_5e4_EDL_original"

# os.makedirs(models_path, exist_ok=True)
# log_file = os.path.join(models_path, "training_log.csv")

# BEST_MACRO_F1 = -1.0
# for epoch in range(1, NUM_EPOCHS + 1):
    
#     train_loss, train_auc, train_acc, train_evidences, train_targets, train_correct, train_preds, train_unc, train_unc_stats = train_one_epoch_edl(model, train_loader, optimizer, epoch)
#     val_loss, val_auc, val_acc, val_evidences, val_targets, val_correct, val_probs, val_unc, val_unc_stats, val_metrics = validate_edl(model, val_loader, epoch)
#     test_loss, test_auc, test_acc, test_evidences, test_targets, test_correct, test_probs, test_unc, test_unc_stats, test_metrics = test_edl(model, test_loader, epoch)

#     agg_train_unc = aggregate_uncertainty_stats(train_unc_stats)
#     agg_val_unc   = aggregate_uncertainty_stats(val_unc_stats)
#     agg_test_unc = aggregate_uncertainty_stats(test_unc_stats)

#     train_unc_history.append(agg_train_unc)
#     val_unc_history.append(agg_val_unc)
#     test_unc_history.append(agg_test_unc)

#     train_ev_mean = np.mean(train_evidences)
#     val_ev_mean = np.mean(val_evidences)
#     test_ev_mean = np.mean(test_evidences)

#     train_ev_history.append(train_ev_mean)
#     val_ev_history.append(val_ev_mean)
#     test_ev_history.append(test_ev_mean)

#     if not math.isnan(val_auc):
#         scheduler.step(val_auc)

#     # compute mean evidence for successes and failures (train)
#     train_evidences = np.array(train_evidences)
#     train_correct = np.array(train_correct)
#     if train_evidences.size > 0:
#         ev_succ = float(np.mean(train_evidences[train_correct == True])) if np.any(train_correct == True) else 0.0
#         ev_fail = float(np.mean(train_evidences[train_correct == False])) if np.any(train_correct == False) else 0.0
#     else:
#         ev_succ, ev_fail = 0.0, 0.0

#     # compute for val
#     val_evidences = np.array(val_evidences)
#     val_correct = np.array(val_correct)
#     if val_evidences.size > 0:
#         ev_succ_val = float(np.mean(val_evidences[val_correct == True])) if np.any(val_correct == True) else 0.0
#         ev_fail_val = float(np.mean(val_evidences[val_correct == False])) if np.any(val_correct == False) else 0.0
#     else:
#         ev_succ_val, ev_fail_val = 0.0, 0.0

#     test_evidences = np.array(test_evidences)
#     test_correct = np.array(test_correct)
#     if test_evidences.size > 0:
#         ev_succ_test = float(np.mean(test_evidences[test_correct == True])) if np.any(test_correct == True) else 0.0
#         ev_fail_test = float(np.mean(test_evidences[test_correct == False])) if np.any(test_correct == False) else 0.0
#     else:
#         ev_succ_test, ev_fail_test = 0.0, 0.0

#     L_train_acc.append(train_acc)
#     L_train_ev_s.append(ev_succ)
#     L_train_ev_f.append(ev_fail)
#     L_val_acc.append(val_acc)
#     L_val_ev_s.append(ev_succ_val)
#     L_val_ev_f.append(ev_fail_val)
#     L_test_acc.append(test_acc)
#     L_test_ev_s.append(ev_succ_test)
#     L_test_ev_f.append(ev_fail_test)

#     print_epoch_summary(
#         epoch,
#         train_loss,
#         val_loss,
#         test_loss,
#         train_auc,
#         val_auc,
#         test_auc,
#         train_acc,
#         val_acc,
#         test_acc,
#         test_metrics,
#         test_unc
#     )

#     row = {
#         "epoch": epoch,
#         "train_loss": train_loss, "val_loss": val_loss, "test_loss": test_loss,
#         "train_acc": train_acc, "val_acc": val_acc, "test_acc": test_acc,
#         "train_auc": train_auc, "val_auc": val_metrics["roc_auc"], "test_auc": test_metrics["roc_auc"],
#         "val_pr_auc": val_metrics["pr_auc"], "test_pr_auc": test_metrics["pr_auc"], 
#         "val_f1": val_metrics["f1"],  "test_f1": test_metrics["f1"], "val_macro_f1": val_metrics["macro_f1"], "test_macro_f1": test_metrics["macro_f1"],
#         "val_precision": val_metrics["precision"], "val_recall": val_metrics["recall"], "val_npv": val_metrics["npv"],
#         "test_precision": test_metrics["precision"], "test_recall": test_metrics["recall"], "test_npv": test_metrics["npv"],
#         "val_specificity": val_metrics["specificity"], "test_specificity": test_metrics["specificity"], 
#         "val_sens_at_spec_90": val_metrics['sens_at_spec'], "test_sens_at_spec_90": test_metrics['sens_at_spec'],
#         "avg_entropy": agg_test_unc['avg_entropy'], "entropy_std": agg_test_unc['entropy_std'],
#         "avg_uncertainty": agg_test_unc['avg_uncertainty'], "uncertainty_std": agg_test_unc['uncertainty_std'],
#         "entropy_class0_avg": agg_test_unc['entropy_class0_avg'], "entropy_class0_std": agg_test_unc['entropy_class0_std'], "entropy_class1_avg": agg_test_unc['entropy_class1_avg'], 
#         "entropy_class1_std": agg_test_unc['entropy_class1_std'], "uncertainty_class0_avg": agg_test_unc['uncertainty_class0_avg'], "uncertainty_class0_std": agg_test_unc['uncertainty_class0_std'], 
#         "uncertainty_class1_avg": agg_test_unc['uncertainty_class1_avg'], "uncertainty_class1_std": agg_test_unc['uncertainty_class1_std'],
#         "tn": test_metrics["tn"], "fp": test_metrics["fp"], "fn": test_metrics["fn"], "tp": test_metrics["tp"], "n_samples": test_metrics["n_samples"],
#     }

#     append_metrics_to_csv(log_file, row)

#     ckpt = {
#         "epoch": epoch,
#         "state_dict": model.state_dict(),
#         "optimizer": optimizer.state_dict(),
#         "val_auc": test_metrics["macro_f1"]
#     }

#     torch.save(ckpt, os.path.join(models_path, f"epoch_{epoch}.pth"))

#     if test_metrics["macro_f1"] > BEST_MACRO_F1:
#         BEST_MACRO_F1 = test_metrics["macro_f1"]
#         torch.save(ckpt, os.path.join(models_path, "best_model.pth"))

In [28]:
import pandas as pd

# models_path = "ResNet34_neh_reduced_5e4_EDL_original_finetune_ucsd"
models_path = "Five_percent_20p_ResNet34_neh_reduced_5e4_EDL_original_finetune_dhu"


os.makedirs(models_path, exist_ok=True)
log_file = os.path.join(models_path, "training_log.csv")

checkpoint_path = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL/epoch_3.pth"

checkpoint = torch.load(checkpoint_path, map_location=DEVICE)

model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE)


NUM_EPOCHS = 20

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,              # higher LR is safe here
    weight_decay=1e-4
)

BEST_MACRO_F1 = -1.0
for epoch in range(1, NUM_EPOCHS + 1):
    
    train_loss, train_auc, train_acc, train_evidences, train_targets, train_correct, train_preds, train_unc, train_unc_stats = train_one_epoch_edl(model, train_loader, optimizer, epoch)
    val_loss, val_auc, val_acc, val_evidences, val_targets, val_correct, val_probs, val_unc, val_unc_stats, val_metrics = validate_edl(model, val_loader, epoch)

    agg_train_unc = aggregate_uncertainty_stats(train_unc_stats)
    agg_val_unc   = aggregate_uncertainty_stats(val_unc_stats)

    train_unc_history.append(agg_train_unc)
    val_unc_history.append(agg_val_unc)

    train_ev_mean = np.mean(train_evidences)
    val_ev_mean = np.mean(val_evidences)

    train_ev_history.append(train_ev_mean)
    val_ev_history.append(val_ev_mean)

    if not math.isnan(val_auc):
        scheduler.step(val_auc)

    # compute mean evidence for successes and failures (train)
    train_evidences = np.array(train_evidences)
    train_correct = np.array(train_correct)
    if train_evidences.size > 0:
        ev_succ = float(np.mean(train_evidences[train_correct == True])) if np.any(train_correct == True) else 0.0
        ev_fail = float(np.mean(train_evidences[train_correct == False])) if np.any(train_correct == False) else 0.0
    else:
        ev_succ, ev_fail = 0.0, 0.0

    # compute for val
    val_evidences = np.array(val_evidences)
    val_correct = np.array(val_correct)
    if val_evidences.size > 0:
        ev_succ_val = float(np.mean(val_evidences[val_correct == True])) if np.any(val_correct == True) else 0.0
        ev_fail_val = float(np.mean(val_evidences[val_correct == False])) if np.any(val_correct == False) else 0.0
    else:
        ev_succ_val, ev_fail_val = 0.0, 0.0


    L_train_acc.append(train_acc)
    L_train_ev_s.append(ev_succ)
    L_train_ev_f.append(ev_fail)
    L_val_acc.append(val_acc)
    L_val_ev_s.append(ev_succ_val)
    L_val_ev_f.append(ev_fail_val)
    

    print_epoch_summary(
        epoch,
        train_loss,
        val_loss,
        0.0,
        train_auc,
        val_auc,
        val_auc,
        train_acc,
        val_acc,
        val_acc,
        val_metrics,
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss, "val_loss": val_loss,
        "train_acc": train_acc, "val_acc": val_acc, 
        "train_auc": train_auc, "val_auc": val_metrics["roc_auc"], 
        "val_pr_auc": val_metrics["pr_auc"],
        "val_f1": val_metrics["f1"],  "val_macro_f1": val_metrics["macro_f1"], 
        "val_precision": val_metrics["precision"], "val_recall": val_metrics["recall"], "val_npv": val_metrics["npv"],
        
        "val_specificity": val_metrics["specificity"],
        "val_sens_at_spec_90": val_metrics['sens_at_spec'], 
    }

    append_metrics_to_csv(log_file, row)
    ckpt = {
        "epoch": epoch,
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "val_auc": val_metrics["roc_auc"]
    }

    torch.save(ckpt, os.path.join(models_path, f"epoch_{epoch}.pth"))

    if val_metrics["macro_f1"] > BEST_MACRO_F1:
        BEST_MACRO_F1 = val_metrics["macro_f1"]
        torch.save(ckpt, os.path.join(models_path, "best_model.pth"))

Epoch 1
------------------------------------------------------------------------------------------
Train | Loss=0.2693  AUC=0.9079  Acc=0.8596
Val   | Loss=0.2282  AUC=0.9429  Acc=0.8966
Test  | Loss=0.0000  AUC=0.9429  Acc=0.8966

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9596  Brier_score=0.1062  F1=0.9032  Macro-F1=0.8961  Precision=0.8990  Recall=0.8952  NPV=0.9231  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=14  FP=2 FN=1  TN=12


Epoch 2
------------------------------------------------------------------------------------------
Train | Loss=0.2304  AUC=0.9215  Acc=0.8860
Val   | Loss=0.2375  AUC=0.9476  Acc=0.8621
Test  | Loss=0.0000  AUC=0.9476  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9476  PR-AUC=0.9616  Brier_score=0.1087  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12


Epoch 3
------------------------------------------------------------------------------------------
Train | Loss=0.2461  AUC=0.9119  Acc=0.8772
Val   | Loss=0.2477  AUC=0.9476  Acc=0.8276
Test  | Loss=0.0000  AUC=0.9476  Acc=0.8276

Classification Metrics:
  ROC-AUC=0.9476  PR-AUC=0.9616  Brier_score=0.1116  F1=0.8276  Macro-F1=0.8276  Precision=0.8286  Recall=0.8286  NPV=0.8000  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=12  FP=2 FN=3  TN=12


Epoch 4
------------------------------------------------------------------------------------------
Train | Loss=0.2296  AUC=0.9178  Acc=0.8772
Val   | Loss=0.2510  AUC=0.9476  Acc=0.8276
Test  | Loss=0.0000  AUC=0.9476  Acc=0.8276

Classification Metrics:
  ROC-AUC=0.9476  PR-AUC=0.9616  Brier_score=0.1112  F1=0.8276  Macro-F1=0.8276  Precision=0.8286  Recall=0.8286  NPV=0.8000  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=12  FP=2 FN=3  TN=12


Epoch 5
------------------------------------------------------------------------------------------
Train | Loss=0.2466  AUC=0.9049  Acc=0.8860
Val   | Loss=0.2535  AUC=0.9476  Acc=0.8621
Test  | Loss=0.0000  AUC=0.9476  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9476  PR-AUC=0.9616  Brier_score=0.1106  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12


...

Epoch 16
------------------------------------------------------------------------------------------
Train | Loss=0.2673  AUC=0.9166  Acc=0.8860
Val   | Loss=0.2707  AUC=0.9429  Acc=0.8621
Test  | Loss=0.0000  AUC=0.9429  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9596  Brier_score=0.1035  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12


Epoch 17
------------------------------------------------------------------------------------------
Train | Loss=0.2677  AUC=0.9280  Acc=0.8684
Val   | Loss=0.2729  AUC=0.9429  Acc=0.8621
Test  | Loss=0.0000  AUC=0.9429  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9596  Brier_score=0.1034  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12


Epoch 18
------------------------------------------------------------------------------------------
Train | Loss=0.2601  AUC=0.9338  Acc=0.8860
Val   | Loss=0.2745  AUC=0.9429  Acc=0.8621
Test  | Loss=0.0000  AUC=0.9429  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9596  Brier_score=0.1031  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12


Epoch 19
------------------------------------------------------------------------------------------
Train | Loss=0.2599  AUC=0.9218  Acc=0.8860
Val   | Loss=0.2746  AUC=0.9429  Acc=0.8621
Test  | Loss=0.0000  AUC=0.9429  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9596  Brier_score=0.1021  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12


Epoch 20
------------------------------------------------------------------------------------------
Train | Loss=0.2549  AUC=0.9335  Acc=0.8860
Val   | Loss=0.2750  AUC=0.9429  Acc=0.8621
Test  | Loss=0.0000  AUC=0.9429  Acc=0.8621

Classification Metrics:
  ROC-AUC=0.9429  PR-AUC=0.9596  Brier_score=0.1013  F1=0.8667  Macro-F1=0.8619  Precision=0.8619  Recall=0.8619  NPV=0.8571  Specificity=0.8571  Sens@Spec90=0.8000 

Confusion Matrix:  TP=13  FP=2 FN=2  TN=12


In [29]:
# raise Exception

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import pickle
from PIL import Image
from tqdm import tqdm
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.isotonic import IsotonicRegression
from typing import Tuple

# =====================================================
# 1. VENN-ABERS CLASS DEFINITION
# =====================================================
class VennAbersBinary:
    def __init__(self):
        self.calib_scores = None
        self.calib_labels = None
        self._fitted = False

    def fit(self, scores: np.ndarray, labels: np.ndarray):
        self.calib_scores = np.asarray(scores, dtype=np.float64)
        self.calib_labels = np.asarray(labels, dtype=np.int32)
        self._fitted = True

    def _fit_isotonic(self, scores, labels):
        ir = IsotonicRegression(out_of_bounds="clip")
        ir.fit(scores, labels)
        return ir

    def predict_interval(self, p: float) -> Tuple[float, float]:
        if not self._fitted:
            raise RuntimeError("Call fit() before predict_interval()")
        scores0 = np.append(self.calib_scores, p)
        labels0 = np.append(self.calib_labels, 0)
        ir0 = self._fit_isotonic(scores0, labels0)
        p0 = float(ir0.predict([p])[0])
        scores1 = np.append(self.calib_scores, p)
        labels1 = np.append(self.calib_labels, 1)
        ir1 = self._fit_isotonic(scores1, labels1)
        p1 = float(ir1.predict([p])[0])
        return min(p0, p1), max(p0, p1)

    def predict(self, p: float, method: str = "mean") -> float:
        p0, p1 = self.predict_interval(p)
        if method == "mean": return 0.5 * (p0 + p1)
        return p0 if method == "lower" else p1

    def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
        return np.array([self.predict(p, method=method) for p in probs])

# =====================================================
# 2. CONFIG & PATHS
# =====================================================
# BEST_CHECKPOINT = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL_original_finetune_octC8/best_model.pth"
# SAVE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL_original_finetune_octC8/venn_abers_fitted.pkl"

BEST_CHECKPOINT = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/Five_percent_20p_ResNet34_neh_reduced_5e4_EDL_original_finetune_dhu/best_model.pth"
SAVE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/Five_percent_20p_ResNet34_neh_reduced_5e4_EDL_original_finetune_dhu/venn_abers_fitted.pkl"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 512
BATCH_SIZE = 32



model = models.resnet34(pretrained=False)
model.fc = nn.Linear(model.fc.in_features, 2)
ckpt = torch.load(BEST_CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt["state_dict"])
model.to(DEVICE).eval()

# =====================================================
# 4. COLLECT VALIDATION PROBABILITIES (EDL)
# =====================================================

val_probs, val_labels = [], []

print(f"Extracting validation scores from: {os.path.basename(BEST_CHECKPOINT)}")
with torch.no_grad():
    for imgs, labels in tqdm(test_loader):
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        
        # EDL Logic
        evidence = F.softplus(logits)
        alpha = evidence + 1
        S = alpha.sum(dim=1, keepdim=True)
        p_hat = alpha / S
        
        val_probs.extend(p_hat[:, 1].cpu().numpy())
        val_labels.extend(labels.numpy())

# =====================================================
# 5. FIT & SAVE VENN-ABERS
# =====================================================
va = VennAbersBinary()
va.fit(scores=np.array(val_probs), labels=np.array(val_labels))

os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
with open(SAVE_PATH, 'wb') as f:
    pickle.dump(va, f)

print(f"\nSuccess! Fitted Venn-Abers model saved to: {SAVE_PATH}")

/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/bharath/.local/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Extracting validation scores from: best_model.pth


100%|██████████| 86/86 [00:06<00:00, 13.51it/s]


Success! Fitted Venn-Abers model saved to: /mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/Five_percent_20p_ResNet34_neh_reduced_5e4_EDL_original_finetune_dhu/venn_abers_fitted.pkl


In [31]:
from sklearn.metrics import f1_score

f1_macro = f1_score(val_labels, (np.array(val_probs) >= 0.5).astype(int), average="macro")
f1_macro


0.8389502815154117

In [32]:
from sklearn.metrics import (
    f1_score, accuracy_score, roc_auc_score,
    precision_score, recall_score,
    confusion_matrix, cohen_kappa_score, fbeta_score
)

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp + 1e-8)

def npv_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fn + 1e-8)

def set_size_and_singleton(y_proba, tau=0.9):
    y_proba = np.asarray(y_proba, dtype=np.float64)
    confidence = np.maximum(y_proba, 1.0 - y_proba)
    singleton_mask = confidence >= tau
    set_sizes = np.where(singleton_mask, 1, 2)
    return {
        "avg_set_size": float(np.mean(set_sizes)),
        "singleton_rate": float(np.mean(singleton_mask)) * 100.0,
    }

# def expected_calibration_error(y_true, y_proba, n_bins=10):
#     y_true = np.asarray(y_true, dtype=int)
#     y_proba = np.asarray(y_proba, dtype=np.float64)
#     y_pred = (y_proba >= 0.5).astype(int)
#     confidence = np.where(y_pred == 1, y_proba, 1.0 - y_proba)
#     bins = np.linspace(0.0, 1.0, n_bins + 1)
#     bin_ids = np.digitize(confidence, bins) - 1
#     ece = 0.0
#     for b in range(n_bins):
#         mask = bin_ids == b
#         if not np.any(mask): continue
#         acc_b = np.mean(y_pred[mask] == y_true[mask])
#         conf_b = np.mean(confidence[mask])
#         ece += np.abs(acc_b - conf_b) * np.mean(mask)
#     return float(ece)


def expected_calibration_error(y_true, y_proba, n_bins=15):
    """
    Standard ECE for binary classification (Guo et al., 2017)
    """
    y_true = np.asarray(y_true, dtype=int)
    y_proba = np.asarray(y_proba, dtype=np.float64)

    # Prediction and confidence
    y_pred = (y_proba >= 0.5).astype(int)
    conf = np.maximum(y_proba, 1.0 - y_proba)

    # Bin edges: confidence ∈ [0.5, 1.0]
    bin_edges = np.linspace(0.5, 1.0, n_bins + 1)

    ece = 0.0
    n = len(y_true)

    for i in range(n_bins):
        bin_lower = bin_edges[i]
        bin_upper = bin_edges[i + 1]

        mask = (conf > bin_lower) & (conf <= bin_upper)
        if not np.any(mask):
            continue

        acc_bin = np.mean(y_pred[mask] == y_true[mask])
        conf_bin = np.mean(conf[mask])
        ece += (np.sum(mask) / n) * abs(acc_bin - conf_bin)

    return float(ece)

def get_ordered_metrics(y_true, y_prob):
    y_pred = (y_prob >= 0.5).astype(int)
    conf_stats = set_size_and_singleton(y_prob, tau=0.9)
    
    # Strictly following the requested order
    return {
        "F1 Macro": f1_score(y_true, y_pred, average="macro"),
        "F2 Macro": fbeta_score(y_true, y_pred, beta=2.0, average="macro", zero_division=0),
        "F2 Weighted": fbeta_score(y_true, y_pred, beta=2.0, average="weighted", zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "AUC": roc_auc_score(y_true, y_prob),
        "Specificity": specificity_score(y_true, y_pred),
        "Precision Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_true, y_pred, average="macro"),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Sensitivity": recall_score(y_true, y_pred),
        "NPV": npv_score(y_true, y_pred),
        "Kappa": cohen_kappa_score(y_true, y_pred),
        "ECE": expected_calibration_error(y_true, y_prob),
        "Set Size": conf_stats["avg_set_size"],
        "Singleton": conf_stats["singleton_rate"]
    }

# Ensure inputs are numpy arrays
y_true_val = np.array(val_labels)
y_prob_val = np.array(val_probs)

# Using your logic to get the dictionary in the specific requested order
val_metrics = get_ordered_metrics(y_true_val, y_prob_val)

# =====================================================
# 2. PRINT RESULTS IN THE SPECIFIED ORDER
# =====================================================
print("\n" + "=" * 55)
print(f"{'Metric':<30} | {'Validation Value':<15}")
print("=" * 55)

for key, value in val_metrics.items():
    suffix = "%" if key == "Singleton" else ""
    print(f"{key:<30} | {value:<15.4f}{suffix}")

print("=" * 55)


Metric                         | Validation Value
F1 Macro                       | 0.8390         
F2 Macro                       | 0.8376         
F2 Weighted                    | 0.8387         
Accuracy                       | 0.8406         
AUC                            | 0.9282         
Specificity                    | 0.7539         
Precision Macro                | 0.8506         
Recall Macro                   | 0.8391         
Precision                      | 0.7955         
Sensitivity                    | 0.9242         
NPV                            | 0.9057         
Kappa                          | 0.6801         
ECE                            | 0.0575         
Set Size                       | 1.2590         
Singleton                      | 74.0999        %


In [33]:
raise Exception

Exception: 

# 50 percent 20 epochs

In [ ]:
# # UCSD 

# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.9335         
# F2 Macro                       | 0.9340         
# F2 Weighted                    | 0.9335         
# Accuracy                       | 0.9336         
# AUC                            | 0.9816         
# Specificity                    | 0.9525         
# Precision Macro                | 0.9332         
# Recall Macro                   | 0.9346         
# Precision                      | 0.9558         
# Sensitivity                    | 0.9167         
# NPV                            | 0.9106         
# Kappa                          | 0.8670         
# ECE                            | 0.0603         
# Set Size                       | 1.4345         
# Singleton                      | 56.5471        %
# =======================================================

In [ ]:
# # DHU 

# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.8278         
# F2 Macro                       | 0.8270         
# F2 Weighted                    | 0.8292         
# Accuracy                       | 0.8295         
# AUC                            | 0.9062         
# Specificity                    | 0.7894         
# Precision Macro                | 0.8299         
# Recall Macro                   | 0.8267         
# Precision                      | 0.8262         
# Sensitivity                    | 0.8640         
# NPV                            | 0.8336         
# Kappa                          | 0.6558         
# ECE                            | 0.0646         
# Set Size                       | 1.2606         
# Singleton                      | 73.9427        %
# =======================================================

In [ ]:
# # oct c8
# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.8768         
# F2 Macro                       | 0.8774         
# F2 Weighted                    | 0.8765         
# Accuracy                       | 0.8768         
# AUC                            | 0.9518         
# Specificity                    | 0.9121         
# Precision Macro                | 0.8779         
# Recall Macro                   | 0.8784         
# Precision                      | 0.9135         
# Sensitivity                    | 0.8447         
# NPV                            | 0.8424         
# Kappa                          | 0.7539         
# ECE                            | 0.0478         
# Set Size                       | 1.4632         
# Singleton                      | 53.6824        %
# =======================================================

# 10 percent 20 epochs

In [ ]:
# # UCSD
# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.9133         
# F2 Macro                       | 0.9132         
# F2 Weighted                    | 0.9135         
# Accuracy                       | 0.9135         
# AUC                            | 0.9711         
# Specificity                    | 0.9068         
# Precision Macro                | 0.9134         
# Recall Macro                   | 0.9132         
# Precision                      | 0.9150         
# Sensitivity                    | 0.9196         
# NPV                            | 0.9118         
# Kappa                          | 0.8266         
# ECE                            | 0.0470         
# Set Size                       | 1.4332         
# Singleton                      | 56.6849        %
# =======================================================

In [ ]:
# # DHU
# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.8391         
# F2 Macro                       | 0.8391         
# F2 Weighted                    | 0.8388         
# Accuracy                       | 0.8394         
# AUC                            | 0.9206         
# Specificity                    | 0.7939         
# Precision Macro                | 0.8423         
# Recall Macro                   | 0.8398         
# Precision                      | 0.8083         
# Sensitivity                    | 0.8857         
# NPV                            | 0.8763         
# Kappa                          | 0.6790         
# ECE                            | 0.0758         
# Set Size                       | 1.1993         
# Singleton                      | 80.0666        %
# =======================================================

In [ ]:
# # oct c8
# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.8804         
# F2 Macro                       | 0.8806         
# F2 Weighted                    | 0.8805         
# Accuracy                       | 0.8806         
# AUC                            | 0.9517         
# Specificity                    | 0.8864         
# Precision Macro                | 0.8802         
# Recall Macro                   | 0.8808         
# Precision                      | 0.8944         
# Sensitivity                    | 0.8752         
# NPV                            | 0.8660         
# Kappa                          | 0.7608         
# ECE                            | 0.0580         
# Set Size                       | 1.6554         
# Singleton                      | 34.4580        %
# =======================================================

# 5 percent Results

In [ ]:
# UCSD 

# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.9083         
# F2 Macro                       | 0.9087         
# F2 Weighted                    | 0.9083         
# Accuracy                       | 0.9084         
# AUC                            | 0.9679         
# Specificity                    | 0.9219         
# Precision Macro                | 0.9081         
# Recall Macro                   | 0.9090         
# Precision                      | 0.9270         
# Sensitivity                    | 0.8962         
# NPV                            | 0.8891         
# Kappa                          | 0.8166         
# ECE                            | 0.0454         
# Set Size                       | 1.3913         
# Singleton                      | 60.8720        %
# =======================================================

In [ ]:
# # DHU

# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.8390         
# F2 Macro                       | 0.8376         
# F2 Weighted                    | 0.8387         
# Accuracy                       | 0.8406         
# AUC                            | 0.9282         
# Specificity                    | 0.7539         
# Precision Macro                | 0.8506         
# Recall Macro                   | 0.8391         
# Precision                      | 0.7955         
# Sensitivity                    | 0.9242         
# NPV                            | 0.9057         
# Kappa                          | 0.6801         
# ECE                            | 0.0575         
# Set Size                       | 1.2590         
# Singleton                      | 74.0999        %
# =======================================================

In [ ]:
# # OCT c8

# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.8349         
# F2 Macro                       | 0.8354         
# F2 Weighted                    | 0.8349         
# Accuracy                       | 0.8350         
# AUC                            | 0.9218         
# Specificity                    | 0.8542         
# Precision Macro                | 0.8351         
# Recall Macro                   | 0.8359         
# Precision                      | 0.8604         
# Sensitivity                    | 0.8175         
# NPV                            | 0.8099         
# Kappa                          | 0.6701         
# ECE                            | 0.0521         
# Set Size                       | 1.5897         
# Singleton                      | 41.0252        %
# =======================================================

# 10 percent Results

In [ ]:
# OCT C8

# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.8752         
# F2 Macro                       | 0.8756         
# F2 Weighted                    | 0.8753         
# Accuracy                       | 0.8753         
# AUC                            | 0.9470         
# Specificity                    | 0.8897         
# Precision Macro                | 0.8752         
# Recall Macro                   | 0.8760         
# Precision                      | 0.8957         
# Sensitivity                    | 0.8622         
# NPV                            | 0.8546         
# Kappa                          | 0.7505         
# ECE                            | 0.0505         
# Set Size                       | 1.6934         
# Singleton                      | 30.6574        %
# =======================================================

In [ ]:
# DHU

# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.8320         
# F2 Macro                       | 0.8319         
# F2 Weighted                    | 0.8316         
# Accuracy                       | 0.8323         
# AUC                            | 0.9131         
# Specificity                    | 0.7807         
# Precision Macro                | 0.8360         
# Recall Macro                   | 0.8328         
# Precision                      | 0.7983         
# Sensitivity                    | 0.8849         
# NPV                            | 0.8736         
# Kappa                          | 0.6649         
# ECE                            | 0.0634         
# Set Size                       | 1.2522         
# Singleton                      | 74.7815        %
# =======================================================

In [ ]:
# # UCSD

# =======================================================
# Metric                         | Validation Value
# =======================================================
# F1 Macro                       | 0.9078         
# F2 Macro                       | 0.9080         
# F2 Weighted                    | 0.9079         
# Accuracy                       | 0.9079         
# AUC                            | 0.9677         
# Specificity                    | 0.9121         
# Precision Macro                | 0.9076         
# Recall Macro                   | 0.9081         
# Precision                      | 0.9182         
# Sensitivity                    | 0.9041         
# NPV                            | 0.8971         
# Kappa                          | 0.8156         
# ECE                            | 0.0421         
# Set Size                       | 1.3606         
# Singleton                      | 63.9436        %
# =======================================================

In [ ]:
# EDl + Venn Abers

In [ ]:
# import os
# import torch
# import numpy as np
# import pandas as pd
# import pickle
# from PIL import Image
# from tqdm import tqdm
# import torch.nn as nn
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from torchvision import models, transforms
# from sklearn.isotonic import IsotonicRegression
# from typing import Tuple

# # =====================================================
# # 1. VENN-ABERS CLASS DEFINITION
# # =====================================================
# class VennAbersBinary:
#     def __init__(self):
#         self.calib_scores = None
#         self.calib_labels = None
#         self._fitted = False

#     def fit(self, scores: np.ndarray, labels: np.ndarray):
#         self.calib_scores = np.asarray(scores, dtype=np.float64)
#         self.calib_labels = np.asarray(labels, dtype=np.int32)
#         self._fitted = True

#     def _fit_isotonic(self, scores, labels):
#         ir = IsotonicRegression(out_of_bounds="clip")
#         ir.fit(scores, labels)
#         return ir

#     def predict_interval(self, p: float) -> Tuple[float, float]:
#         if not self._fitted:
#             raise RuntimeError("Call fit() before predict_interval()")
#         scores0 = np.append(self.calib_scores, p)
#         labels0 = np.append(self.calib_labels, 0)
#         ir0 = self._fit_isotonic(scores0, labels0)
#         p0 = float(ir0.predict([p])[0])
#         scores1 = np.append(self.calib_scores, p)
#         labels1 = np.append(self.calib_labels, 1)
#         ir1 = self._fit_isotonic(scores1, labels1)
#         p1 = float(ir1.predict([p])[0])
#         return min(p0, p1), max(p0, p1)

#     def predict(self, p: float, method: str = "mean") -> float:
#         p0, p1 = self.predict_interval(p)
#         if method == "mean": return 0.5 * (p0 + p1)
#         return p0 if method == "lower" else p1

#     def predict_batch(self, probs: np.ndarray, method: str = "mean") -> np.ndarray:
#         return np.array([self.predict(p, method=method) for p in probs])

# # =====================================================
# # 2. CONFIG & PATHS
# # =====================================================
# BEST_CHECKPOINT = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL/epoch_3.pth"
# VAL_CSV_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/neh_val_reduced.csv"
# SAVE_PATH = "/mnt/8b4bbd12-99b7-4ef1-9218-be56afd51a3d/UCSD_2/ResNet34_neh_reduced_5e4_EDL/venn_abers_fitted.pkl"

# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# IMG_SIZE = 512
# BATCH_SIZE = 32

# # =====================================================
# # 3. DATASET & MODEL PREP
# # =====================================================
# test_transform = transforms.Compose([
#     transforms.Resize((IMG_SIZE, IMG_SIZE)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
# ])

# class UCSDDataset(Dataset):
#     def __init__(self, csv_path, transform):
#         self.df = pd.read_csv(csv_path)
#         self.transform = transform
#     def __len__(self): return len(self.df)
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         img = Image.open(row["new_file_path"]).convert("RGB")
#         label = int(row["binary_label"])
#         return self.transform(img), label

# val_loader = DataLoader(UCSDDataset(VAL_CSV_PATH, test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

# model = models.resnet34(pretrained=False)
# model.fc = nn.Linear(model.fc.in_features, 2)
# ckpt = torch.load(BEST_CHECKPOINT, map_location=DEVICE)
# model.load_state_dict(ckpt["state_dict"])
# model.to(DEVICE).eval()

# # =====================================================
# # 4. COLLECT VALIDATION PROBABILITIES (EDL)
# # =====================================================
# val_probs, val_labels = [], []

# print(f"Extracting validation scores from: {os.path.basename(BEST_CHECKPOINT)}")
# with torch.no_grad():
#     for imgs, labels in tqdm(val_loader):
#         imgs = imgs.to(DEVICE)
#         logits = model(imgs)
        
#         # EDL Logic
#         evidence = F.softplus(logits)
#         alpha = evidence + 1
#         S = alpha.sum(dim=1, keepdim=True)
#         p_hat = alpha / S
        
#         val_probs.extend(p_hat[:, 1].cpu().numpy())
#         val_labels.extend(labels.numpy())

# # =====================================================
# # 5. FIT & SAVE VENN-ABERS
# # =====================================================
# va = VennAbersBinary()
# va.fit(scores=np.array(val_probs), labels=np.array(val_labels))

# os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
# with open(SAVE_PATH, 'wb') as f:
#     pickle.dump(va, f)

# print(f"\nSuccess! Fitted Venn-Abers model saved to: {SAVE_PATH}")